In [0]:
# Databricks notebook source
"""
Buzzmetrix topic tagging notebook for Databricks model serving endpoints.

This version follows the same style as your AI insight notebook:
- manual config block
- `ENDPOINT` driven inference
- retry / rate limit controls
- Spark read / write to Delta tables

It translates each memo into Korean and assigns one reason topic for
positive / negative reviews. Very generic reviews are mapped to
`전반적긍정` or `전반적부정`.
"""

from __future__ import annotations

import json
import os
import re
import time
from typing import Any, Dict, Iterator, List, Optional, Tuple

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T


# =========================
# Manual Config
# =========================
SNAPSHOT_QUARTER = "2026Q1"
PROMPT_VERSION = "v6.1.0"
ENDPOINT = "databricks-gpt-5-2"
RUN_SCOPE = "PROD"

TEST_MODE = False
TEST_MAX_GROUPS = 30
RUN_FULL_AFTER_TEST = False

TEST_CATE_1_DEPTH = "07. 스마트 사용성"
TEST_CATE_2_DEPTH = "07-06. 리모컨 사용성"

RATE_LIMIT_SECONDS = 0.6
MAX_TOKENS = 1600
MAX_RETRIES = 4
BACKOFF_BASE = 1.8

GROUP_DIMS_TO_RUN: List[str] = []  # [] 이면 전체
TARGET_Y_FEATURE = ""  # "" 또는 None 이면 전체

PVAL_MAX = 0.10
ABS_COEF_THRESHOLD = 0.10
USE_IS_DRIVER = True

MAX_DRIVERS_PER_KEY_FULL = 4
MAX_DRIVERS_PER_KEY_COMPACT = 3
MAX_DIFF_STATS_FULL = 10

SOURCE_TABLE = "kic_data_ods.buzzmetrix.buzzmetrix"
SUMMARY_TABLE = f"sandbox.z_jungryo_lee.voc_topic_summary_{PROMPT_VERSION.replace('.', '_')}"
DETAIL_TABLE = f"sandbox.z_jungryo_lee.voc_topic_detail_{PROMPT_VERSION.replace('.', '_')}"
FAILED_TABLE = f"sandbox.z_jungryo_lee.voc_topic_failed_{PROMPT_VERSION.replace('.', '_')}"
TEST_SUMMARY_TABLE = f"{SUMMARY_TABLE}_test"
TEST_DETAIL_TABLE = f"{DETAIL_TABLE}_test"
TEST_FAILED_TABLE = f"{FAILED_TABLE}_test"
TOPIC_MAP_TABLE = None  # optional later mapping table
MAX_FAILED_RETRIES = 2

REMOTE_USABILITY_LV1_FAMILIES = {
    "레이아웃/키": [
        "레이아웃",
        "배치",
        "버튼",
        "버튼배치",
        "키",
        "숫자키",
        "핵심버튼",
        "버튼 위치",
        "복잡한 조작",
        "직관성",
        "navigation",
        "layout",
        "key",
        "button",
    ],
    "포인터": [
        "포인터",
        "pointer",
        "cursor",
        "커서",
        "민감도",
        "감도",
        "air mouse",
        "모션",
        "움직임",
        "반응",
    ],
    "디자인": [
        "디자인",
        "design",
        "백라이트",
        "backlight",
        "조명",
        "light",
        "저가",
        "구형",
        "품질",
        "마감",
        "무게",
        "형태",
        "색상",
    ],
}

REMOTE_USABILITY_OVERALL_LV1 = "전반적리모컨사용성"

REMOTE_USABILITY_LV2_FAMILIES = {
    "버튼 구성이 편리함": [
        "간단",
        "간결",
        "간편",
        "편의",
        "편리",
        "직관",
        "쉬움",
        "사용편의",
        "구성편의",
        "조작성",
        "버튼구성",
        "버튼 구성",
    ],
    "버튼 배치가 난해함": [
        "배치",
        "난해",
        "복잡",
        "혼란",
        "헷갈",
        "불편",
        "어려움",
        "비직관",
        "배열",
        "구성",
    ],
    "핵심 버튼이 부재함": [
        "핵심버튼",
        "필수버튼",
        "주요버튼",
        "부재",
        "없음",
        "없다",
        "미존재",
        "중요버튼",
    ],
    "숫자키가 부재함": [
        "숫자키",
        "number key",
        "num key",
        "numkey",
        "키패드",
    ],
    "포인터 조작이 비호감임": [
        "비호감",
        "불편",
        "거슬",
        "어색",
        "조작감",
        "움직임",
        "사용감",
        "포인터조작",
        "pointer",
        "cursor",
    ],
    "포인터 민감도에 불만이 있음": [
        "민감도",
        "감도",
        "너무 민감",
        "둔감",
        "반응",
        "response",
    ],
    "백라이트가 과도하게 밝음": [
        "백라이트",
        "backlight",
        "빛",
        "조명",
        "야간",
        "밤",
        "어두움",
        "너무 밝",
    ],
    "저가형처럼 품질이 낮음": [
        "저가",
        "구형",
        "품질",
        "싸구려",
        "cheap",
        "낡",
        "마감",
        "재질",
        "저품질",
    ],
}

# =========================
# Taxonomy Policy
# =========================
# 카테고리별 리뷰 다양성에 따라 taxonomy 상한선을 다르게 둡니다.
# 아래 값은 "주제 사전을 설계할 때의 목표 상한선"이며,
# 실제 태깅 로직은 1개 주제만 반환합니다.
#
# 수정 위치:
# 1) 리뷰가 단순한 카테고리면 LOW / MEDIUM를 더 엄격하게
# 2) 리뷰가 복합적인 카테고리면 HIGH를 더 넓게
# 3) 특정 cate_1_depth만 예외가 있으면 CATEGORY_TOPIC_OVERRIDE에 추가
CATEGORY_COMPLEXITY_RULES = {
    "LOW": {
        "max_topic_lv1": 6,
        "max_topic_lv2_per_lv1": 5,
        "total_topic_budget": 30,
    },
    "MEDIUM": {
        "max_topic_lv1": 10,
        "max_topic_lv2_per_lv1": 10,
        "total_topic_budget": 80,
    },
    "HIGH": {
        "max_topic_lv1": 12,
        "max_topic_lv2_per_lv1": 15,
        "total_topic_budget": 150,
    },
}

# 카테고리별 예외 상한선.
# 키는 cate_1_depth 기준이며, 필요한 경우 cate_2_depth까지 포함해서 더 세분화해도 됩니다.
CATEGORY_TOPIC_OVERRIDE = {
    # 예시:
    # "07. 스마트 사용성": {"complexity": "HIGH", "max_topic_lv1": 12, "max_topic_lv2_per_lv1": 15},
    # "02. 화질": {"complexity": "HIGH", "max_topic_lv1": 12, "max_topic_lv2_per_lv1": 18},
}

# 자동 생성 주제의 기본 정리 규칙.
# - 너무 일반적인 리뷰는 전반적긍정/전반적부정으로 수렴
# - 상세 주제는 대주제 1개당 대체로 5~15개 정도를 목표
DEFAULT_TOPIC_POLICY = {
    "overall_positive_label": "전반적긍정",
    "overall_negative_label": "전반적부정",
    "neutral_label": "중립/기타",
    "recommended_lv1_range": (5, 12),
    "recommended_lv2_range_per_lv1": (5, 15),
    "hard_lv1_cap": 12,
    "hard_lv2_cap": 20,
}

print(f"[CONFIG] quarter={SNAPSHOT_QUARTER}, prompt_version={PROMPT_VERSION}, endpoint={ENDPOINT}, run_scope={RUN_SCOPE}")
print(f"[CONFIG] dims={GROUP_DIMS_TO_RUN}, target_y_feature={TARGET_Y_FEATURE}, test_mode={TEST_MODE} (limit={TEST_MAX_GROUPS})")
print(f"[CONFIG] pval<={PVAL_MAX}, abs(coef)>={ABS_COEF_THRESHOLD}, use_is_driver={USE_IS_DRIVER}")


# =========================
# Helpers
# =========================

GENERIC_POSITIVE = {
    "good",
    "great",
    "nice",
    "love it",
    "love",
    "awesome",
    "amazing",
    "excellent",
    "perfect",
    "happy",
    "satisfied",
    "very good",
    "very nice",
    "top notch",
    "좋아요",
    "좋다",
    "좋네요",
    "좋음",
    "굿",
    "최고",
    "만족",
    "만족합니다",
    "괜찮아요",
    "아주 좋아요",
    "훌륭",
}

GENERIC_NEGATIVE = {
    "bad",
    "terrible",
    "awful",
    "poor",
    "disappointed",
    "not good",
    "hate",
    "broken",
    "issue",
    "problem",
    "불만",
    "별로",
    "나쁨",
    "안좋아요",
    "안 좋아요",
    "문제",
    "고장",
    "최악",
}

# 전반적긍정/부정으로 보내는 아주 짧은 리뷰의 상한선
# 이 값은 매우 보수적으로 잡아, 웬만한 표현은 세부 분류로 가게 한다.
OVERALL_MAX_TOKENS = 1
OVERALL_MAX_CHARS = 8


def clean_text(text: Any) -> str:
    if text is None:
        return ""
    return str(text).replace("\n", " ").replace("\r", " ").strip()


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def normalize_sentiment(raw: Any) -> str:
    text = clean_text(raw)
    if text in {"긍정", "positive", "pos"}:
        return "긍정"
    if text in {"부정", "negative", "neg"}:
        return "부정"
    if text in {"중립", "중립/질문", "neutral", "question", "neutral/question"}:
        return "중립"
    return "기타"


def overall_topic(sentiment: str) -> Tuple[str, str]:
    if sentiment == "긍정":
        return "전반적긍정", "전반적긍정"
    if sentiment == "부정":
        return "전반적부정", "전반적부정"
    return "중립/기타", "중립/기타"


def get_remote_lv1_candidates() -> List[str]:
    return list(REMOTE_USABILITY_LV1_FAMILIES.keys())


def infer_remote_lv1(text: str, topic_lv2: str = "") -> str:
    """lv2와 memo를 바탕으로 리모컨 사용성의 lv1 가족 분류를 후처리한다."""
    if not topic_lv2:
        return REMOTE_USABILITY_OVERALL_LV1

    haystack = normalize_whitespace(f"{text} {topic_lv2}".lower())
    best_label = REMOTE_USABILITY_OVERALL_LV1
    best_score = 0

    for label, keywords in REMOTE_USABILITY_LV1_FAMILIES.items():
        score = sum(1 for kw in keywords if kw.lower() in haystack)
        if score > best_score:
            best_label = label
            best_score = score

    return best_label


def normalize_remote_lv2(text: str) -> str:
    """lv2를 문장형 표준 라벨로 묶는다."""
    candidate = normalize_whitespace(str(text)).replace(" ", "")
    candidate = re.sub(r"[^\w가-힣]+", "", candidate)
    if not candidate:
        return "기타 개선이 필요함"

    special_labels = {
        "전반적긍정",
        "전반적부정",
        "중립/기타",
        "분류실패",
    }
    if candidate in special_labels:
        return candidate

    candidate_phrase = normalize_whitespace(str(text))
    if candidate_phrase.endswith("함") and len(candidate_phrase) <= 25:
        return candidate_phrase

    haystack = normalize_whitespace(f"{candidate} {candidate_phrase}".lower())
    best_label = None
    best_score = 0
    for label, keywords in REMOTE_USABILITY_LV2_FAMILIES.items():
        score = sum(1 for kw in keywords if kw.lower() in haystack)
        if score > best_score:
            best_label = label
            best_score = score

    if best_label:
        return best_label

    if len(candidate_phrase) <= 20:
        return f"{candidate_phrase}이/가 관련됨"

    return "기타 개선이 필요함"


def canonical_lv1_label(cate_1_depth: str, cate_2_depth: str, memo: str = "", topic_lv1: str = "", topic_lv2: str = "") -> str:
    """lv1 기본 라벨을 카테고리별로 정한다.

    - 리모컨 사용성은 상위 개념(레이아웃/키, 포인터, 디자인)으로 정규화
    - 그 외 카테고리는 대분류명을 유지
    """
    if cate_2_depth == TEST_CATE_2_DEPTH:
        return infer_remote_lv1(memo, topic_lv2)
    return cate_1_depth


def extract_category_keywords(cate_1_depth: str, cate_2_depth: str) -> List[str]:
    """카테고리명에서 전반적 평가 판정에 쓸 키워드를 추출한다."""
    raw_parts = [cate_1_depth, cate_2_depth]
    keywords: List[str] = []

    for raw in raw_parts:
        if not raw:
            continue
        cleaned = re.sub(r"^\*+", "", str(raw))
        cleaned = re.sub(r"^\d{2}(?:-\d{2})?\.\s*", "", cleaned)
        cleaned = re.sub(r"\s*\(.+?\)\s*", " ", cleaned)
        cleaned = cleaned.replace("/", " ")
        cleaned = cleaned.replace("-", " ")
        cleaned = cleaned.replace(".", " ")
        for token in cleaned.split():
            token = token.strip()
            if len(token) >= 2:
                keywords.append(token)

    return sorted(set(keywords), key=len, reverse=True)


def neutral_integrated_label(cate_1_depth: str, cate_2_depth: str, memo: str) -> str:
    """중립 리뷰용 통합 주제 라벨 생성."""
    memo_hint = clean_text(memo)
    system = (
        "You are a multilingual VOC analyst for TV reviews. "
        "The sentiment is neutral. Translate the review into Korean first, then infer a single integrated topic "
        "for this subcategory without splitting into level-1 and level-2. "
        "Return only valid JSON with the exact keys: `memo_ko`, `integrated_topic`, `confidence`. "
        "Rules: "
        "1) `memo_ko` must be a natural Korean translation of the original review. "
        "2) `integrated_topic` must be a single Korean label, max 20 characters. "
        "3) Make the label specific to the subcategory context. "
        "4) Do not create a hierarchical topic. "
        "5) If the review is too generic, use `중립/기타`. "
    )
    user = (
        f"cate_1_depth: {cate_1_depth}\n"
        f"cate_2_depth: {cate_2_depth}\n"
        f"memo: {memo_hint}\n\n"
        "Output JSON only."
    )
    result = call_databricks_endpoint(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
    )
    memo_ko = clean_text(result.get("memo_ko", "")) or memo_hint
    topic = clean_text(result.get("integrated_topic", "")) or "중립/기타"
    return memo_ko, topic


def get_topic_policy(cate_1_depth: str) -> Dict[str, Any]:
    """카테고리별로 적용할 taxonomy 상한선을 반환한다.

    수정 포인트:
    - 특정 대분류를 더 넓게/좁게 보고 싶으면 CATEGORY_TOPIC_OVERRIDE를 수정
    - 기본 상한선은 DEFAULT_TOPIC_POLICY 및 CATEGORY_COMPLEXITY_RULES에서 조정
    """
    override = CATEGORY_TOPIC_OVERRIDE.get(cate_1_depth)
    if override:
        complexity = override.get("complexity", "MEDIUM")
        base = dict(CATEGORY_COMPLEXITY_RULES.get(complexity, CATEGORY_COMPLEXITY_RULES["MEDIUM"]))
        base.update(override)
        return base

    return dict(CATEGORY_COMPLEXITY_RULES["MEDIUM"])


def format_policy_for_prompt(cate_1_depth: str) -> str:
    policy = get_topic_policy(cate_1_depth)
    return (
        f"Category policy for {cate_1_depth}: "
        f"max_topic_lv1={policy['max_topic_lv1']}, "
        f"max_topic_lv2_per_lv1={policy['max_topic_lv2_per_lv1']}, "
        f"total_topic_budget={policy['total_topic_budget']}. "
        f"Use {DEFAULT_TOPIC_POLICY['overall_positive_label']} and "
        f"{DEFAULT_TOPIC_POLICY['overall_negative_label']} for very generic reviews. "
        f"Use {DEFAULT_TOPIC_POLICY['neutral_label']} for neutral/question reviews."
    )


def print_taxonomy_policy_preview() -> None:
    """노트북에서 정책을 확인할 때 쓰는 미리보기."""
    print("[TAXONOMY POLICY]")
    print(f"  overall_positive_label = {DEFAULT_TOPIC_POLICY['overall_positive_label']}")
    print(f"  overall_negative_label = {DEFAULT_TOPIC_POLICY['overall_negative_label']}")
    print(f"  neutral_label          = {DEFAULT_TOPIC_POLICY['neutral_label']}")
    print(f"  recommended_lv1_range  = {DEFAULT_TOPIC_POLICY['recommended_lv1_range']}")
    print(f"  recommended_lv2_range  = {DEFAULT_TOPIC_POLICY['recommended_lv2_range_per_lv1']}")
    print(f"  hard_lv1_cap           = {DEFAULT_TOPIC_POLICY['hard_lv1_cap']}")
    print(f"  hard_lv2_cap           = {DEFAULT_TOPIC_POLICY['hard_lv2_cap']}")
    print("[CATEGORY OVERRIDES]")
    for k, v in CATEGORY_TOPIC_OVERRIDE.items():
        print(f"  {k}: {v}")


print_taxonomy_policy_preview()


def should_use_overall_topic(text: str, sentiment: str, cate_1_depth: str, cate_2_depth: str) -> bool:
    if not text:
        return True

    lowered = normalize_whitespace(text.lower())
    compact = re.sub(r"[^\w가-힣]+", " ", lowered).strip()
    compact_no_space = re.sub(r"\s+", "", compact)
    token_count = len([t for t in compact.split(" ") if t])
    char_count = len(re.sub(r"\s+", "", text))
    category_keywords = extract_category_keywords(cate_1_depth, cate_2_depth)

    if sentiment == "긍정" and (lowered in GENERIC_POSITIVE or compact in GENERIC_POSITIVE or compact_no_space in GENERIC_POSITIVE):
        return True
    if sentiment == "부정" and (lowered in GENERIC_NEGATIVE or compact in GENERIC_NEGATIVE or compact_no_space in GENERIC_NEGATIVE):
        return True

    # 카테고리명만 붙어서 짧게 평가한 경우는 전반적 라벨로 보낸다.
    if sentiment in {"긍정", "부정"} and token_count <= 4 and char_count <= 20:
        if any(keyword and keyword in compact for keyword in category_keywords):
            return True

    # 아주 짧고 단순한 경우만 전반적 라벨로 보낸다.
    if token_count <= OVERALL_MAX_TOKENS and char_count <= OVERALL_MAX_CHARS and sentiment in {"긍정", "부정"}:
        return True

    return False


def build_messages(cate_1_depth: str, cate_2_depth: str, sentiment: str, memo: str) -> List[Dict[str, str]]:
    system = (
        "You are a multilingual VOC analyst for TV reviews. "
        "Translate the review into Korean first, then infer the most specific reason topic. "
        "Return only valid JSON with the exact keys: "
        "`memo_ko`, `topic_lv1`, `topic_lv2`, `is_overall`, `confidence`. "
        "Rules: "
        "1) `memo_ko` must be a natural Korean translation of the original review. "
        "2) `topic_lv2` must be a specific detailed reason label in Korean, written in a short sentence-like Korean style that naturally ends with `함` when possible. "
        "3) `topic_lv1` will be derived later, so set it to `__DERIVE__` or leave it generic. "
        "4) If the review is too generic or just a short overall opinion like good/great/좋아요, "
        "set `is_overall` to true and use `전반적긍정` or `전반적부정` for both topic labels. "
        "5) Do not invent unsupported details. "
        "6) Prefer feature-reason labels that fit the category context, not sentiment words. "
        f"7) {format_policy_for_prompt(cate_1_depth)} "
    )

    user = (
        f"cate_1_depth: {cate_1_depth}\n"
        f"cate_2_depth: {cate_2_depth}\n"
        f"sentiment: {sentiment}\n"
        f"memo: {memo}\n\n"
        "Output JSON only."
    )
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]


def extract_json_object(text: str) -> Dict[str, Any]:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        return json.loads(match.group(0))
    raise ValueError(f"Model did not return valid JSON: {text[:500]}")


def call_databricks_endpoint(messages: List[Dict[str, str]]) -> Dict[str, Any]:
    from mlflow.deployments import get_deploy_client

    payload = {
        "messages": messages,
        "temperature": 0.0,
        "max_tokens": MAX_TOKENS,
    }

    db_client = get_deploy_client("databricks")

    last_error: Optional[Exception] = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = db_client.predict(endpoint=ENDPOINT, inputs=payload)

            if isinstance(resp, dict):
                if "choices" in resp and resp["choices"]:
                    content = resp["choices"][0]["message"]["content"]
                    return extract_json_object(content)
                if "predictions" in resp and resp["predictions"]:
                    pred0 = resp["predictions"][0]
                    if isinstance(pred0, dict) and "content" in pred0:
                        return extract_json_object(pred0["content"])
                    if isinstance(pred0, str):
                        return extract_json_object(pred0)
                if "content" in resp:
                    return extract_json_object(resp["content"])

            if isinstance(resp, str):
                return extract_json_object(resp)

            raise ValueError(f"Unexpected endpoint response schema: {type(resp)}")
        except Exception as exc:
            last_error = exc
            sleep_s = BACKOFF_BASE ** attempt
            time.sleep(sleep_s)

    raise RuntimeError(f"Databricks endpoint request failed after retries: {last_error}")


def tag_one_row(
    cate_1_depth: str,
    cate_2_depth: str,
    sentiment_raw: str,
    memo: str,
    attempt_no: int,
) -> Dict[str, Any]:
    sentiment = normalize_sentiment(sentiment_raw)
    memo_clean = clean_text(memo)

    try:
        if sentiment == "중립":
            memo_ko, integrated_topic = neutral_integrated_label(cate_1_depth, cate_2_depth, memo_clean)
            normalized_lv2 = normalize_remote_lv2(integrated_topic)
            lv1_family = infer_remote_lv1(memo_clean, integrated_topic) if cate_2_depth == TEST_CATE_2_DEPTH else canonical_lv1_label(cate_1_depth, cate_2_depth, memo_clean, "", integrated_topic)
            return {
                "attempt_no": attempt_no,
                "_row_status": "success",
                "topic_mode": "neutral_integrated",
                "memo_ko": memo_ko,
                "topic_lv1_auto": lv1_family,
                "topic_lv2_auto": normalized_lv2,
                "topic_lv1_final": lv1_family,
                "topic_lv2_final": normalized_lv2,
                "topic_label_final": f"{lv1_family} / {normalized_lv2}",
                "is_overall": True,
                "confidence": 0.85,
                "topic_source": "dbx_endpoint",
                "tagger": "dbx_endpoint",
                "error_message": None,
            }

        if should_use_overall_topic(memo_clean, sentiment, cate_1_depth, cate_2_depth):
            lv1, lv2 = overall_topic(sentiment)
            lv1_family = REMOTE_USABILITY_OVERALL_LV1 if cate_2_depth == TEST_CATE_2_DEPTH else cate_1_depth
            return {
                "attempt_no": attempt_no,
                "_row_status": "success",
                "topic_mode": "overall",
                "memo_ko": memo_clean,
                "topic_lv1_auto": lv1_family,
                "topic_lv2_auto": normalize_remote_lv2(lv2),
                "topic_lv1_final": lv1_family,
                "topic_lv2_final": normalize_remote_lv2(lv2),
                "topic_label_final": f"{lv1_family} / {normalize_remote_lv2(lv2)}",
                "is_overall": True,
                "confidence": 0.95,
                "topic_source": "heuristic",
                "tagger": "heuristic_overall",
                "error_message": None,
            }

        messages = build_messages(cate_1_depth, cate_2_depth, sentiment, memo_clean)
        result = call_databricks_endpoint(messages)
        time.sleep(RATE_LIMIT_SECONDS)

        memo_ko = clean_text(result.get("memo_ko", ""))
        topic_lv1 = clean_text(result.get("topic_lv1", ""))
        topic_lv2 = clean_text(result.get("topic_lv2", ""))
        is_overall = bool(result.get("is_overall", False))
        confidence = float(result.get("confidence", 0.7) or 0.7)

        if not memo_ko:
            memo_ko = memo_clean
        if not topic_lv2:
            topic_lv2 = topic_lv1 if topic_lv1 and topic_lv1 != "__DERIVE__" else "분류실패"

        topic_lv2 = normalize_remote_lv2(topic_lv2)

        if cate_2_depth == TEST_CATE_2_DEPTH:
            topic_lv1 = infer_remote_lv1(memo_clean, topic_lv2)
        else:
            topic_lv1 = canonical_lv1_label(cate_1_depth, cate_2_depth, memo_clean, topic_lv1, topic_lv2)

        return {
            "attempt_no": attempt_no,
            "_row_status": "success",
            "topic_mode": "hierarchical",
            "memo_ko": memo_ko,
            "topic_lv1_auto": topic_lv1,
            "topic_lv2_auto": topic_lv2,
            "topic_lv1_final": topic_lv1,
            "topic_lv2_final": topic_lv2,
            "topic_label_final": f"{topic_lv1} / {topic_lv2}",
            "is_overall": is_overall,
            "confidence": confidence,
            "topic_source": "dbx_endpoint",
            "tagger": "dbx_endpoint",
            "error_message": None,
        }
    except Exception as exc:
        return {
            "attempt_no": attempt_no,
            "_row_status": "error",
            "topic_mode": "error",
            "memo_ko": memo_clean,
            "topic_lv1_auto": canonical_lv1_label(cate_1_depth, cate_2_depth, memo_clean),
            "topic_lv2_auto": "분류실패",
            "topic_lv1_final": canonical_lv1_label(cate_1_depth, cate_2_depth, memo_clean),
            "topic_lv2_final": "분류실패",
            "topic_label_final": f"{canonical_lv1_label(cate_1_depth, cate_2_depth, memo_clean)} / 분류실패",
            "is_overall": False,
            "confidence": 0.0,
            "topic_source": "error",
            "tagger": "error",
            "error_message": str(exc)[:1000],
        }


def build_tag_partition(attempt_no: int):
    def tag_partition(pdf_iter: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
        for pdf in pdf_iter:
            out_rows: List[Dict[str, Any]] = []
            for row in pdf.itertuples(index=False, name=None):
                cate_1_depth, cate_2_depth, sentiment_raw, memo, row_id = row
                tagged = tag_one_row(
                    cate_1_depth=cate_1_depth,
                    cate_2_depth=cate_2_depth,
                    sentiment_raw=sentiment_raw,
                    memo=memo,
                    attempt_no=attempt_no,
                )
                out_rows.append(
                    {
                        "_row_id": row_id,
                        "cate_1_depth": cate_1_depth,
                        "cate_2_depth": cate_2_depth,
                        "sentiment_raw": sentiment_raw,
                        "sentiment_norm": normalize_sentiment(sentiment_raw),
                        "memo": memo,
                        **tagged,
                    }
                )

            yield pd.DataFrame(
                out_rows,
                columns=[
                    "_row_id",
                    "cate_1_depth",
                    "cate_2_depth",
                    "sentiment_raw",
                    "sentiment_norm",
                    "memo",
                    "attempt_no",
                    "_row_status",
                    "topic_mode",
                    "memo_ko",
                    "topic_lv1_auto",
                    "topic_lv2_auto",
                    "topic_lv1_final",
                    "topic_lv2_final",
                    "topic_label_final",
                    "is_overall",
                    "confidence",
                    "topic_source",
                    "tagger",
                    "error_message",
                ],
            )

    return tag_partition


def run_tagging_pass(source_df: DataFrame, attempt_no: int) -> DataFrame:
    schema = T.StructType(
        [
            T.StructField("_row_id", T.LongType(), True),
            T.StructField("cate_1_depth", T.StringType(), True),
            T.StructField("cate_2_depth", T.StringType(), True),
            T.StructField("sentiment_raw", T.StringType(), True),
            T.StructField("sentiment_norm", T.StringType(), True),
            T.StructField("memo", T.StringType(), True),
            T.StructField("attempt_no", T.IntegerType(), True),
            T.StructField("_row_status", T.StringType(), True),
            T.StructField("topic_mode", T.StringType(), True),
            T.StructField("memo_ko", T.StringType(), True),
            T.StructField("topic_lv1_auto", T.StringType(), True),
            T.StructField("topic_lv2_auto", T.StringType(), True),
            T.StructField("topic_lv1_final", T.StringType(), True),
            T.StructField("topic_lv2_final", T.StringType(), True),
            T.StructField("topic_label_final", T.StringType(), True),
            T.StructField("is_overall", T.BooleanType(), True),
            T.StructField("confidence", T.DoubleType(), True),
            T.StructField("topic_source", T.StringType(), True),
            T.StructField("tagger", T.StringType(), True),
            T.StructField("error_message", T.StringType(), True),
        ]
    )

    return source_df.mapInPandas(build_tag_partition(attempt_no), schema=schema)


def load_source_df(test_mode: bool = False) -> DataFrame:
    df = spark.sql(
        f"""
        SELECT DISTINCT
            cate_1_depth,
            cate_2_depth,
            sentiment,
            memo
        FROM {SOURCE_TABLE}
        WHERE memo IS NOT NULL
        """
    )

    if GROUP_DIMS_TO_RUN:
        # Optional filter hook if you want to run only specific category groups.
        # Expected formats:
        # - ["01. 사이즈|01-01. TV 사이즈", ...]
        # - or custom values you adapt here.
        allowed = GROUP_DIMS_TO_RUN
        df = df.withColumn("group_key", F.concat_ws("|", F.col("cate_1_depth"), F.col("cate_2_depth")))
        df = df.where(F.col("group_key").isin(allowed)).drop("group_key")

    if TARGET_Y_FEATURE:
        df = df.where(F.col("cate_2_depth") == F.lit(TARGET_Y_FEATURE))

    if test_mode:
        df = (
            df.where(F.col("cate_1_depth") == F.lit(TEST_CATE_1_DEPTH))
            .where(F.col("cate_2_depth") == F.lit(TEST_CATE_2_DEPTH))
            .orderBy("sentiment", "memo")
            .limit(TEST_MAX_GROUPS)
        )

    return df.withColumn("_row_id", F.monotonically_increasing_id())


def load_topic_map_df() -> Optional[DataFrame]:
    if not TOPIC_MAP_TABLE:
        return None
    if not spark.catalog.tableExists(TOPIC_MAP_TABLE):
        return None
    return spark.table(TOPIC_MAP_TABLE)


def apply_topic_mapping(df: DataFrame, mapping_df: Optional[DataFrame]) -> DataFrame:
    if mapping_df is None:
        return (
            df.withColumn("topic_lv1_final", F.col("topic_lv1_auto"))
            .withColumn("topic_lv2_final", F.col("topic_lv2_auto"))
            .withColumn("topic_source", F.col("tagger"))
            .withColumn("topic_label_final", F.concat_ws(" / ", F.col("topic_lv1_final"), F.col("topic_lv2_final")))
        )

    mapping = mapping_df.select(
        "cate_1_depth",
        "cate_2_depth",
        "sentiment_norm",
        "topic_lv1_auto",
        "topic_lv2_auto",
        "topic_lv1_final",
        "topic_lv2_final",
    )

    joined = df.join(
        F.broadcast(mapping),
        on=[
            "cate_1_depth",
            "cate_2_depth",
            "sentiment_norm",
            "topic_lv1_auto",
            "topic_lv2_auto",
        ],
        how="left",
    )

    return (
        joined.withColumn("_mapping_hit", F.col("topic_lv1_final").isNotNull() | F.col("topic_lv2_final").isNotNull())
        .withColumn("topic_lv1_final", F.coalesce(F.col("topic_lv1_final"), F.col("topic_lv1_auto")))
        .withColumn("topic_lv2_final", F.coalesce(F.col("topic_lv2_final"), F.col("topic_lv2_auto")))
        .withColumn("topic_source", F.when(F.col("_mapping_hit"), F.lit("mapped")).otherwise(F.col("tagger")))
        .withColumn("topic_label_final", F.concat_ws(" / ", F.col("topic_lv1_final"), F.col("topic_lv2_final")))
        .drop("_mapping_hit")
    )


def assemble_output_df(source_df: DataFrame, attempt_no: int) -> DataFrame:
    tagged_df = run_tagging_pass(source_df, attempt_no=attempt_no)
    final_df = apply_topic_mapping(tagged_df, load_topic_map_df())

    return final_df.select(
        "_row_id",
        "cate_1_depth",
        "cate_2_depth",
        "sentiment_raw",
        "sentiment_norm",
        "memo",
        "attempt_no",
        "_row_status",
        "topic_mode",
        "memo_ko",
        "topic_lv1_auto",
        "topic_lv2_auto",
        "topic_lv1_final",
        "topic_lv2_final",
        "topic_label_final",
        "is_overall",
        "confidence",
        "topic_source",
        "error_message",
    )


def retry_failed_rows(source_with_id_df: DataFrame, failed_df: DataFrame, attempt_no: int) -> DataFrame:
    failed_ids = failed_df.select("_row_id").distinct()
    return source_with_id_df.join(failed_ids, on="_row_id", how="inner").select(
        "cate_1_depth",
        "cate_2_depth",
        "sentiment",
        "memo",
        "_row_id",
    )


def build_category_topic_share_df(detail_df: DataFrame) -> DataFrame:
    from pyspark.sql import Window

    topic_df = (
        detail_df.where(F.col("_row_status") == "success")
        .groupBy(
            "cate_1_depth",
            "cate_2_depth",
            "sentiment_norm",
            "topic_lv1_final",
            "topic_lv2_final",
            "topic_label_final",
        )
        .agg(
            F.count("*").alias("row_cnt"),
            F.avg("confidence").alias("avg_confidence"),
        )
    )

    w = Window.partitionBy("cate_1_depth", "cate_2_depth")
    topic_df = (
        topic_df.withColumn("category_total_cnt", F.sum("row_cnt").over(w))
        .withColumn("row_share", F.round(F.col("row_cnt") / F.col("category_total_cnt"), 4))
        .withColumn("row_share_pct", F.round(F.col("row_share") * 100, 2))
        .orderBy(F.desc("row_cnt"), F.asc("cate_1_depth"), F.asc("cate_2_depth"))
    )

    return topic_df


def save_output(df: DataFrame, detail_table: str, summary_table: str, failed_table: str) -> Tuple[DataFrame, DataFrame, DataFrame]:
    detail_df = df.cache()
    summary_df = build_category_topic_share_df(detail_df)
    failed_df = detail_df.where(F.col("_row_status") == "error")

    if spark.catalog.tableExists(detail_table):
        print(f"[SAVE] overwrite -> {detail_table}")
        detail_df.write.mode("overwrite").format("delta").saveAsTable(detail_table)
    else:
        print(f"[SAVE] create -> {detail_table}")
        detail_df.write.mode("overwrite").format("delta").saveAsTable(detail_table)

    if spark.catalog.tableExists(summary_table):
        print(f"[SAVE] overwrite -> {summary_table}")
        summary_df.write.mode("overwrite").format("delta").saveAsTable(summary_table)
    else:
        print(f"[SAVE] create -> {summary_table}")
        summary_df.write.mode("overwrite").format("delta").saveAsTable(summary_table)

    if failed_df.take(1):
        if spark.catalog.tableExists(failed_table):
            print(f"[SAVE] overwrite -> {failed_table}")
            failed_df.write.mode("overwrite").format("delta").saveAsTable(failed_table)
        else:
            print(f"[SAVE] create -> {failed_table}")
            failed_df.write.mode("overwrite").format("delta").saveAsTable(failed_table)

    detail_df.unpersist()
    return detail_df, summary_df, failed_df


def run_pipeline(test_mode: bool, detail_table: str, summary_table: str, failed_table: str) -> Dict[str, Any]:
    source_df = load_source_df(test_mode=test_mode)

    remaining_source = source_df
    collected_success: Optional[DataFrame] = None
    final_failed_df: Optional[DataFrame] = None

    for attempt_no in range(1, MAX_FAILED_RETRIES + 2):
        print(f"[RUN] attempt_no={attempt_no}, rows={remaining_source.count()}")
        detail_df = assemble_output_df(remaining_source, attempt_no=attempt_no).cache()

        success_df = detail_df.where(F.col("_row_status") == "success")
        failed_df = detail_df.where(F.col("_row_status") == "error")

        collected_success = success_df if collected_success is None else collected_success.unionByName(success_df)
        final_failed_df = failed_df

        if not failed_df.take(1):
            detail_df.unpersist()
            break

        if attempt_no >= MAX_FAILED_RETRIES + 1:
            detail_df.unpersist()
            break

        retry_source = retry_failed_rows(remaining_source, failed_df, attempt_no=attempt_no + 1)
        remaining_source = retry_source
        detail_df.unpersist()

    if collected_success is None and final_failed_df is None:
        raise RuntimeError("No rows were processed. Check source filters and test mode settings.")

    if collected_success is None:
        collected_success = final_failed_df.limit(0)

    if final_failed_df is None:
        final_failed_df = collected_success.limit(0)

    final_detail_df = collected_success.unionByName(final_failed_df).cache()

    detail_saved, summary_saved, failed_saved = save_output(
        final_detail_df,
        detail_table=detail_table,
        summary_table=summary_table,
        failed_table=failed_table,
    )

    display(summary_saved.toPandas())

    if failed_saved.take(1):
        display(failed_saved.toPandas())

    final_detail_df.unpersist()

    return {
        "detail_df": detail_saved,
        "summary_df": summary_saved,
        "failed_df": failed_saved,
    }


# =========================
# Run
# =========================

print("[RUN] test pass first")
test_result = run_pipeline(
    test_mode=True,
    detail_table=TEST_DETAIL_TABLE,
    summary_table=TEST_SUMMARY_TABLE,
    failed_table=TEST_FAILED_TABLE,
)

if RUN_FULL_AFTER_TEST:
    print("[RUN] full pass")
    full_result = run_pipeline(
        test_mode=False,
        detail_table=DETAIL_TABLE,
        summary_table=SUMMARY_TABLE,
        failed_table=FAILED_TABLE,
    )


In [0]:
print("[RUN] full pass")
full_result = run_pipeline(
    test_mode=False,
    detail_table=DETAIL_TABLE,
    summary_table=SUMMARY_TABLE,
    failed_table=FAILED_TABLE,
)